# Delta Lake Liquid Clustering — Step-by-Step Field Guide

This guide captures the complete workflow for enabling, troubleshooting, and maintaining **Liquid Clustering** on Databricks Unity Catalog / Delta Lake tables, using `nyctaxi.02_silver.yellow_trips_cleansed` as the reference implementation.

---

## 1. Inspect Table Metadata (Partition & Cluster Check)

Before applying clustering, check if the table currently uses traditional Hive partitioning or clustering.

```python
# Inspect format, partitions, and clustering configuration
details = spark.sql("DESCRIBE DETAIL nyctaxi.`02_silver`.yellow_trips_cleansed")
display(details.select("format", "partitionColumns", "clusteringColumns"))
```

### How to interpret the output:
- **`partitionColumns: []`**: Table is unpartitioned.
- **`partitionColumns: ['date', ...]`**: Table is partitioned. You must replace the partitioning scheme.
- **`clusteringColumns: ['col1', 'col2']`**: Liquid Clustering is active on these columns.

---

## 2. Enable Liquid Clustering

Liquid clustering replaces traditional Hive partitioning. You can define up to **4 clustering columns**.

### Option A: Explicit Clustering Keys (Recommended)
Choose high-cardinality columns frequently queried in `WHERE`, `JOIN`, or `GROUP BY` conditions.

```python
# For unpartitioned tables:
spark.sql("""
    ALTER TABLE nyctaxi.`02_silver`.yellow_trips_cleansed 
    CLUSTER BY (tpep_pickup_datetime, pu_location_id)
""")
```

```python
# For previously partitioned tables:
spark.sql("""
    ALTER TABLE nyctaxi.`02_silver`.yellow_trips_cleansed 
    REPLACE PARTITIONED BY WITH CLUSTER BY (tpep_pickup_datetime, pu_location_id)
""")
```

---

### Option B: Auto Clustering (`CLUSTER BY AUTO`)
Databricks can dynamically pick clustering keys based on query history (Databricks Runtime 15.2+).

```python
spark.sql("""
    ALTER TABLE nyctaxi.`02_silver`.yellow_trips_cleansed 
    CLUSTER BY AUTO
""")
```

> **Why did `clusteringColumns` show `[]` after `CLUSTER BY AUTO`?**  
> `CLUSTER BY AUTO` does **not** persist a static column list into `clusteringColumns`. Instead, Databricks evaluates query logs dynamically and records its decisions in the transaction commit metrics (`clusteringMetrics`). If explicit visibility in `DESCRIBE DETAIL` is desired, use explicit keys.

---

## 3. Reorganize Historical Files (`OPTIMIZE FULL`)

The `ALTER TABLE` statement only affects future writes. To physically rewrite existing data according to the clustering keys, execute an optimization job:

```python
# Reorganize all historical data files
spark.sql("OPTIMIZE nyctaxi.`02_silver`.yellow_trips_cleansed FULL")
```

- **`OPTIMIZE ... FULL`**: Rewrites all data files to conform to clustering keys (use on initial setup or schema changes).
- **`OPTIMIZE ...` (without FULL)**: Fast, incremental optimization that clusters only unorganized/appended data files.

---

## 4. Verify Active Clustering Keys

Verify that the table properties have updated:

```python
details = spark.sql("DESCRIBE DETAIL nyctaxi.`02_silver`.yellow_trips_cleansed")
display(details.select("format", "partitionColumns", "clusteringColumns"))
```

**Expected verification output:**
| format | partitionColumns | clusteringColumns |
| :--- | :--- | :--- |
| `delta` | `[]` | `["tpep_pickup_datetime", "pu_location_id"]` |

---

## 5. Safe Write Patterns (Preserving Clustering Metadata)

### ❌ Anti-Pattern: Standard DataFrame Overwrite
```python
# DANGER: Drops the table metadata and wipes clusteringColumns!
df.write.mode("overwrite").saveAsTable("nyctaxi.`02_silver`.yellow_trips_cleansed")
```

---

### Pattern 1: Overwrite with `overwriteSchema = false`
Overwrites the data content while keeping the existing table definition, Delta features, and clustering keys intact:

```python
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "false")
      .saveAsTable("nyctaxi.`02_silver`.yellow_trips_cleansed")
)
```

---

### Pattern 2: Explicit Clustering via `DataFrameWriterV2`
If recreating or creating tables from DataFrames, specify `clusterBy` directly:

```python
(
    df.writeTo("nyctaxi.`02_silver`.yellow_trips_cleansed")
      .clusterBy("tpep_pickup_datetime", "pu_location_id")
      .createOrReplace()
)
```

---

### Pattern 3: Incremental Ingestion via Delta MERGE
Avoid full table rewrites by upserting new records:

```python
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "nyctaxi.`02_silver`.yellow_trips_cleansed")

(
    target.alias("t")
    .merge(
        source=df.alias("s"),
        condition="""
            t.tpep_pickup_datetime = s.tpep_pickup_datetime 
            AND t.pu_location_id = s.pu_location_id
            AND t.vendor_id = s.vendor_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# Run standard OPTIMIZE after the merge
spark.sql("OPTIMIZE nyctaxi.`02_silver`.yellow_trips_cleansed")
```

---

## Summary Checklist
- [x] Check partition & clustering status (`DESCRIBE DETAIL`).
- [x] Define clustering columns (`ALTER TABLE ... CLUSTER BY (...)`).
- [x] Recluster existing files (`OPTIMIZE ... FULL`).
- [x] Confirm columns appear in `clusteringColumns`.
- [x] Use `.option("overwriteSchema", "false")` or Delta `MERGE` to prevent accidental resets.

In [0]:
# see the partition 
details = spark.sql("DESCRIBE DETAIL nyctaxi.`02_silver`.yellow_trips_cleansed")
display(details.select("format", "partitionColumns", "clusteringColumns"))

In [0]:
from pyspark.sql.functions import col, timestamp_diff, broadcast

In [0]:
df_yt = spark.read.table('nyctaxi.`02_silver`.yellow_trips_cleansed')
df_lookup = spark.read.table('nyctaxi.`02_silver`.taxi_zone_lookup')

# 1. Prepare two aliased copies of lookup to prevent column name collisions
lookup_pu = (
    df_lookup
    .select(
        col("locationId").alias("pu_loc_key"),
        col("borough").alias("pu_borough"),
        col("zone").alias("pu_zone")
    )
)

lookup_do = (
    df_lookup
    .select(
        col("locationId").alias("do_loc_key"),
        col("borough").alias("do_borough"),
        col("zone").alias("do_zone")
    )
)

In [0]:
df_yt = df_yt.withColumn(
            'trip_duration_mins', timestamp_diff('MINUTE',col('tpep_pickup_datetime'), col('tpep_dropoff_datetime'))
        )

In [0]:
# Get total number of in-memory Spark partitions
num_partitions = df_yt.rdd.getNumPartitions()
print(f"Number of Spark partitions: {num_partitions}")


In [0]:

df_enriched = df_yt.join(
                    broadcast(lookup_pu),
                    on = df_yt.pu_location_id == lookup_pu.pu_loc_key,
                    how = "left"
                ).join(
                    broadcast(lookup_do),
                    on = df_yt.do_location_id == lookup_do.do_loc_key,
                    how = "left"
                )

enriched_columns = [
    "vendor", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_mins",
    "passenger_count", "trip_distance", "rate_type", "store_and_fwd_flag",
    "pu_borough", "do_borough", "pu_zone", "do_zone",
    "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee",
    "cbd_congestion_fee", "processed_timestamp"
]

df_enriched = df_enriched.select(enriched_columns)

In [0]:
(
    df_enriched.writeTo("nyctaxi.`02_silver`.yellow_trips_enriched")
                .clusterBy("tpep_pickup_datetime", "pu_location_id")
                .createOrReplace()
)

# PySpark Joins on Databricks: Revision Notes

Quick notes for revision. Written for managed Delta tables on Databricks (Spark 4.x / DBR 17+).

---

## 1. Types of joins

### 1.1 Logical join types (what rows you get)

| Join type | `how=` | What you get | Typical use |
|---|---|---|---|
| Inner | `inner` | Only rows that match on both sides | Normal enrichment |
| Left outer | `left` | All left rows, nulls where no match | Keep all facts, add lookup info |
| Right outer | `right` | All right rows, nulls where no match | Rare, usually flip it to a left join |
| Full outer | `full` | All rows from both sides | Reconciliation, finding mismatches |
| Left semi | `left_semi` | Left rows that HAVE a match. Only left columns | "Does it exist?" filter, cheaper than inner + distinct |
| Left anti | `left_anti` | Left rows that have NO match. Only left columns | "Find missing", e.g. orders with no payment |
| Cross | `cross` | Every left row with every right row (A x B) | Small grids only. Dangerous on big data |

```python
from pyspark.sql import functions as F

orders = spark.table("main.sales.orders")
cust   = spark.table("main.sales.customers")

orders.join(cust, "customer_id", "inner")      # matches only
orders.join(cust, "customer_id", "left")       # keep all orders
orders.join(cust, "customer_id", "left_semi")  # orders that have a customer
orders.join(cust, "customer_id", "left_anti")  # orders with NO customer
orders.crossJoin(cust)                         # explicit cross join
```

Notes:
- `left_semi` and `left_anti` never duplicate left rows. Inner joins can (if the right side has duplicate keys).
- If both tables have the same column name and you join with a condition (`a.id == b.id`), you get two `id` columns. Join with the column name (`"id"`) to get one, or alias the DataFrames.
- Spark SQL also supports `NATURAL JOIN` and `USING (col)`. Prefer explicit keys.

### 1.2 Physical join strategies (how Spark runs it)

Spark picks one of these. You see the choice in `df.explain()` or the Spark UI / Query Profile.

| Strategy | When Spark uses it | Shuffle? | Speed |
|---|---|---|---|
| **Broadcast Hash Join (BHJ)** | One side is small enough to send to every executor | No shuffle of the big side | Fastest |
| **Shuffle Hash Join (SHJ)** | Equi-join, one side much smaller per partition, builds a hash table | Yes (both sides) | Good, but uses more memory |
| **Sort Merge Join (SMJ)** | Default for two big tables on an equi-join | Yes (both sides) + sort | Reliable, scales, slower than BHJ |
| **Broadcast Nested Loop Join (BNLJ)** | Non-equi condition (`<`, `>`, `between`) with one small side | No big shuffle | Slow if the "small" side is not small |
| **Cartesian Product** | Cross join or non-equi join with no small side | Yes | Worst. Avoid |

Rules of thumb:
- Equi-join (`a.id == b.id`) can use BHJ, SHJ or SMJ. Non-equi cannot use SMJ or SHJ, so it falls to BNLJ or Cartesian.
- **Broadcast threshold:** `spark.sql.autoBroadcastJoinThreshold` defaults to 10 MB. With AQE on Databricks, Spark can also switch to a broadcast join at runtime once it sees the real size of a side (it measures after the shuffle stage).
- AQE can also convert SMJ to SHJ at runtime, split skewed partitions, and coalesce small shuffle partitions.

```python
# Force or suggest a strategy with hints
orders.join(F.broadcast(cust), "customer_id")                 # broadcast
orders.join(cust.hint("shuffle_hash"), "customer_id")         # shuffle hash
orders.join(cust.hint("merge"), "customer_id")                # sort merge
orders.hint("shuffle_replicate_nl").join(cust, "customer_id") # cartesian-style (rare)
```

### 1.3 Special join patterns

| Pattern | What it is | Fix / tool |
|---|---|---|
| **Range / interval join** | `a.ts BETWEEN b.start AND b.end` | Databricks `RANGE_JOIN` hint with a bin size |
| **Skew join** | One key (e.g. `NULL`, `"UNKNOWN"`) has huge data | AQE skew join, or salting |
| **Self join** | Table joined to itself (hierarchies, previous row) | Often a window function (`lag`) is cheaper |
| **Join on array / explode** | Join after exploding an array | Filter before exploding, use higher-order functions |

```python
# Range join hint on Databricks (bin size depends on your data)
events.hint("range_join", 3600).join(
    windows, (events.ts >= windows.start) & (events.ts < windows.end)
)
```

---

## 2. What to consider before joining

Think of it as a pre-flight checklist. Do these before you write the join.

### 2.1 Reduce the data first
1. **Filter early.** Apply `where` on both sides before the join. (Catalyst usually pushes filters down, but not through UDFs or some complex expressions, so do it yourself and check `explain`.)
2. **Select only the columns you need.** Less data to shuffle and sort. Never `select *` into a big join.
3. **Aggregate before joining** if you only need totals (join the summary, not the raw rows).
4. **Deduplicate the lookup side** (`dropDuplicates(["key"])`) so you do not multiply rows by accident.

### 2.2 Know your data
5. **Table sizes.** Is one side small? If yes, broadcast it.
6. **Key uniqueness.** One-to-one, one-to-many or many-to-many? Many-to-many can explode row counts. Check with `count()` vs `countDistinct()` on the key.
7. **Skew.** Are a few keys very heavy? Check the top keys:
   ```python
   df.groupBy("customer_id").count().orderBy(F.desc("count")).show(10)
   ```
8. **Null keys.** `NULL` never equals `NULL` in a normal join, so null-key rows do not match. If you have a huge null group, filter it out first and handle it separately (it can also cause skew).
9. **Data types must match.** Join `int` to `int`, `string` to `string`. A type mismatch can force a cast on the key, which can block pruning and broadcast decisions. Cast on the small side if you must.
10. **Case and whitespace.** `"abc"` and `"ABC "` do not match. Clean keys first (`trim`, `lower`) if the source is messy.

### 2.3 Pick the right join and the right condition
11. **Use an equi-join whenever possible.** Rewrite non-equi joins if you can (for example, add a date bucket column and join on it, then filter).
12. **Use `left_semi` / `left_anti`** instead of inner join + `distinct`, or `NOT IN`. `NOT IN` is also null-unsafe.
13. **Never cross join by accident.** A missing join condition silently becomes a Cartesian product (Spark 3+ errors unless `spark.sql.crossJoin.enabled` is true, which is the default in recent versions, so check your plan).
14. **Avoid UDFs in the join condition.** They block optimisations and force slow paths.

### 2.4 Partitions and shuffle
15. **Shuffle partitions.** On Databricks, AQE with auto-optimised shuffle usually handles this. If AQE is off, `spark.sql.shuffle.partitions` (default 200) may be too low or high.
16. **Do not repartition by habit.** An explicit `repartition("key")` before a join only helps if you reuse that partitioning for several joins, otherwise it is an extra shuffle.
17. **Join order for several tables.** Join the small or most selective tables first so the intermediate result stays small. Check that Spark did not reorder badly.

### 2.5 Always verify
18. Run `df.explain("formatted")` (or `EXPLAIN FORMATTED` in SQL). Look for:
    - `BroadcastHashJoin` vs `SortMergeJoin`
    - `Exchange` (shuffle) operators and how many
    - `PartitionFilters` / `PushedFilters` / dynamic pruning on the scan
19. Open the **Spark UI / Query Profile**: look for spill, skewed tasks (max task time far above median), and shuffle size.
20. Compare **row counts before and after** the join. If it grew unexpectedly, your key is not unique.

### 2.6 Quick decision guide

```
Is one side small (tens of MB, up to a few hundred MB)?
  -> Yes: broadcast it.
  -> No: sort merge join (default). Then check for skew.

Is the condition non-equi (<, >, BETWEEN)?
  -> Try to make it equi (bucket / date key). Else use RANGE_JOIN hint.

Do you only need to know if a match exists / not?
  -> left_semi / left_anti.

Is one key very heavy (NULL, 'UNKNOWN', a mega customer)?
  -> AQE skew join first. If still slow: filter that key out and handle separately, or salt.
```

---

## 3. Your data is in Databricks managed tables: how to optimise joins

Managed Delta tables give you a lot for free. Use it.

### 3.1 Make the tables easy to read and skip files
1. **Liquid clustering on your join keys (and common filter columns).**
   This is the modern replacement for partitioning + `ZORDER`. It lays data out so Spark can skip files and reads are less scattered.
   ```sql
   -- new table
   CREATE TABLE main.sales.orders (...) CLUSTER BY (customer_id, order_date);

   -- existing table
   ALTER TABLE main.sales.orders CLUSTER BY (customer_id, order_date);
   OPTIMIZE main.sales.orders;

   -- let Databricks choose the columns for you
   ALTER TABLE main.sales.orders CLUSTER BY AUTO;
   ```
   Pick keys you actually filter and join on. Do not pick too many columns.
2. **Do not over-partition.** Hive-style partitioning is only for very large tables with a low-cardinality column you always filter on (usually a date). Otherwise prefer liquid clustering.
3. **Compact small files.** Many small files slow every read. Run `OPTIMIZE`, or let **predictive optimization** do it for managed tables (it runs `OPTIMIZE`, `VACUUM` and statistics for you when enabled).
   ```sql
   OPTIMIZE main.sales.orders;
   ```
4. **Keep file sizes healthy.** Managed tables with optimized writes and auto compaction avoid small-file buildup from frequent small writes.

### 3.2 Give the optimiser good information
5. **Keep table statistics fresh.** Better stats mean better join strategy decisions (for example, choosing broadcast). Predictive optimization maintains stats on managed tables. Otherwise:
   ```sql
   ANALYZE TABLE main.sales.customers COMPUTE STATISTICS FOR ALL COLUMNS;
   ```
6. **Delta stats cover the first columns.** By default, file-level min/max stats are collected for the first 32 columns. Put join and filter columns early in the schema, or set `delta.dataSkippingStatsColumns`.
7. **Use the same data type for the key in both tables** at table design time. Fix it in the table, not in every query.

### 3.3 Use the runtime features (mostly automatic)
8. **AQE** (on by default): re-plans after shuffles, so it can switch to broadcast, coalesce shuffle partitions and split skewed partitions. Check it is on:
   ```python
   spark.conf.get("spark.sql.adaptive.enabled")
   spark.conf.get("spark.sql.adaptive.skewJoin.enabled")
   ```
9. **Dynamic file pruning (DFP)** on Delta: when you filter the small (dimension) table, Databricks skips files of the big (fact) table at runtime. It works best when the fact table is clustered or Z-ordered on the join key. Dynamic partition pruning is the same idea for partitioned tables.
10. **Runtime Bloom filter joins:** for a big table joined to a much smaller, filtered one, Spark can build a Bloom filter from the small side and drop non-matching rows from the big side early, before the shuffle.
11. **Photon:** keep the cluster on a Photon-enabled runtime. Joins, aggregations and scans run faster. Avoid Python UDFs in the join path, because they can push work out of Photon.
12. **Disk cache:** repeated reads of the same Delta data benefit from the Databricks disk cache on cluster local SSDs (use cache-accelerated worker types).

### 3.4 Query habits that help
13. **Filter on the clustering / partition column** so files are pruned:
    ```python
    orders = spark.table("main.sales.orders").where("order_date >= '2026-01-01'")
    ```
14. **Broadcast small dimension tables** explicitly if AQE or stats do not choose it and you know the size.
15. **Pre-aggregate** or build a **silver / gold table** for a join you run every day, instead of repeating a heavy join in each notebook. Materialise it once, query it many times.
16. **Fix skew at the source** when you can (clean the `UNKNOWN` / `NULL` keys, split the mega key) instead of fighting it in every query.

### 3.5 What does NOT apply (common confusion)
- **Bucketing** is rarely worth it on Delta and is not supported for Delta tables on Databricks. Use liquid clustering instead.
- **Storage-partitioned joins** (skip the shuffle when both sides share a layout) work for V2 sources such as Iceberg. They do not help for Delta on Databricks.
- `ZORDER BY` still works but is the older approach. A table uses either liquid clustering or partitioning/Z-order, not both.

### 3.6 Checklist for a slow join on managed tables

```
[ ] Filter and select columns BEFORE the join
[ ] Small side broadcast (check explain)
[ ] Tables clustered on the join key (CLUSTER BY / CLUSTER BY AUTO)
[ ] OPTIMIZE has run (or predictive optimization is on)
[ ] Statistics are fresh (ANALYZE TABLE)
[ ] Key types match, nulls handled, duplicates removed
[ ] AQE on, skew join on, Photon on
[ ] Spark UI: no big spill, no one slow task, shuffle size reasonable
[ ] Row count after join is what you expected
```

---

## 4. Quick code reference (copy into a notebook)

```python
from pyspark.sql import functions as F

fact = spark.table("main.sales.orders").where("order_date >= '2026-01-01'") \
            .select("order_id", "customer_id", "amount", "order_date")
dim  = spark.table("main.sales.customers").select("customer_id", "segment") \
            .dropDuplicates(["customer_id"])

# 1) Check key quality
print(dim.count(), dim.select("customer_id").distinct().count())     # should be equal
fact.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)  # skew check

# 2) Join with broadcast on the small side
result = fact.join(F.broadcast(dim), "customer_id", "left")

# 3) Inspect the plan
result.explain("formatted")

# 4) Semi / anti
has_cust = fact.join(dim, "customer_id", "left_semi")
no_cust  = fact.join(dim, "customer_id", "left_anti")
```

```sql
-- Table maintenance for managed tables
ALTER TABLE main.sales.orders CLUSTER BY (customer_id, order_date);
OPTIMIZE main.sales.orders;
ANALYZE TABLE main.sales.customers COMPUTE STATISTICS FOR ALL COLUMNS;
```

---

## 5. Interview one-liners

- **Broadcast vs sort merge?** Broadcast sends the small table to every executor so the big table is not shuffled. Sort merge shuffles and sorts both sides. Use broadcast when one side is small.
- **Semi vs inner join?** Semi only filters the left side and never duplicates rows. Inner returns columns from both and can multiply rows.
- **How do you fix a skewed join?** Let AQE split skewed partitions. If that is not enough: isolate the hot key, or add a salt to the key and replicate the other side.
- **Why is my join slow?** Check the plan and Spark UI for shuffle size, spill, skew, a missed broadcast, a type mismatch or small files.
- **How do managed tables help joins?** Liquid clustering and statistics give better file skipping and join decisions. Predictive optimization keeps files compacted and stats fresh. AQE, dynamic file pruning and Photon do the rest at run time.
- **Why does a null key not match?** `NULL = NULL` is not true in SQL. Use `eqNullSafe` (`<=>`) if you want nulls to match.